In [51]:
# Import Necessary Libraries
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.tree  import plot_tree



print("All libraries Imported!")

All libraries Imported!


In [2]:
df = pd.read_csv("shop_smart_ecommerce.csv")

In [6]:
df.isnull().sum() # Null is zero!

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

In [7]:
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [8]:
features = ["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues", "SpecialDay", "Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
target = ["Revenue"]

In [14]:
#  One hot encoding
ohe = OneHotEncoder(handle_unknown = "ignore", sparse_output=False)
month_encoded = ohe.fit_transform(df[["Month"]])
month_cols = ohe.get_feature_names_out(["Month"])
df[month_cols] = month_encoded
df.drop(columns = ["Month"], inplace = True)

visitorType_encoded = ohe.fit_transform(df[["VisitorType"]])
visitorType_cols = ohe.get_feature_names_out(["VisitorType"])
df[visitorType_cols] = visitorType_encoded
df.drop(columns = ["VisitorType"], inplace = True)

 

df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 29 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Administrative                 12330 non-null  int64  
 1   Administrative_Duration        12330 non-null  float64
 2   Informational                  12330 non-null  int64  
 3   Informational_Duration         12330 non-null  float64
 4   ProductRelated                 12330 non-null  int64  
 5   ProductRelated_Duration        12330 non-null  float64
 6   BounceRates                    12330 non-null  float64
 7   ExitRates                      12330 non-null  float64
 8   PageValues                     12330 non-null  float64
 9   SpecialDay                     12330 non-null  float64
 10  OperatingSystems               12330 non-null  int64  
 11  Browser                        12330 non-null  int64  
 12  Region                         12330 non-null 

In [33]:
X = df.drop(columns = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration", "Revenue"])
y = df["Revenue"]

In [36]:
X.shape

(12330, 24)

In [38]:
y.value_counts(normalize = True)

Revenue
0    0.845255
1    0.154745
Name: proportion, dtype: float64

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

In [43]:
print(y_train.value_counts(normalize = True))
print(y_test.value_counts(normalize = True))

Revenue
0    0.845296
1    0.154704
Name: proportion, dtype: float64
Revenue
0    0.845093
1    0.154907
Name: proportion, dtype: float64


In [48]:
base_model = DecisionTreeClassifier()

base_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [49]:
y_pred = base_model.predict(X_test)

In [62]:
scores = [accuracy_score, recall_score, precision_score, f1_score]
for score in scores:
    print(f"{score.__name__}: {score(y_test, y_pred)}")

accuracy_score: 0.8609083536090836
recall_score: 0.5916230366492147
precision_score: 0.5472154963680388
f1_score: 0.5685534591194968


In [63]:
# Post Pruning after the model
full_tree = DecisionTreeClassifier(random_state = 42)
full_tree.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [65]:
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
# print(ccp_alphas)

In [66]:
#  Training our  model for each alphas
tree = []
for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state = 42, ccp_alpha = alpha)
    model.fit(X_train, y_train)
    tree.append((model, alpha))

best_f1 = 0
best_alpha = 0

for model, alpha in tree:
    y_pred = model.predict(X_test)
    curr_f1 = f1_score(y_test, y_pred)
    if curr_f1 > best_f1:
        best_f1 = curr_f1
        best_alpha = alpha

In [68]:
print(best_f1)

0.6431718061674009


In [69]:
best_model = DecisionTreeClassifier(random_state = 42, ccp_alpha = best_alpha)
best_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [ ]:
plt.figure(figsize = 18, 10))

plot_tree(
    best_mode,
    feature_names = X.columns,
    class_names = ["Purchases", "Not Purchased"],
    filled = True
)
p